In [1]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

Sat Sep 19 16:08:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers datasets huggingface_hub accelerate --quiet

In [3]:
import transformers, datasets
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)

transformers: 5.16.1
datasets: 4.8.5


## Step 2: Sentiment analysis with pipelines

In [4]:
from transformers import pipeline

# 1. Load a sentiment-analysis pipeline (defaults to a small model)
sentiment = pipeline("sentiment-analysis")

# 2. Run inference
examples = [
    "Hugging Face makes NLP super accessible!",
    "I dislike bugs in my code..."
]
results = sentiment(examples)
for text, res in zip(examples, results):
    print(f"{text!r:50} → label={res['label']}, score={res['score']:.3f}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

'Hugging Face makes NLP super accessible!'         → label=POSITIVE, score=1.000
'I dislike bugs in my code...'                     → label=NEGATIVE, score=0.999


## Step 3: Text generation with GPT-2

In [5]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
prompt = "Once upon a time"
out = generator(prompt, max_length=30, num_return_sequences=1)
print(out[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Once upon a time, it would have been possible to have a great deal of freedom in your life. Why should you have to sacrifice your freedom? Let's just say, in the end, it took some effort and sacrifice.

The following is a list of things you can do to prevent this from happening:

Get out of the house:

If you don't want to go to church, you might be forced to stay on your parents' payroll.

If you don't want to go to church, you might be forced to stay on your parents' payroll. Get in the habit of taking part in recreational activities:

If you're really into music, you might want to take a break from making music to let the time pass.

If you're really into music, you might want to take a break from making music to let the time pass. Get out of the habit of drinking:

If you are a member of a religious group that you don't want to go to church, you might prefer not to attend church.

If you are a member of a religious group that you don't want to go to church, you might prefer not to 

## Step 4: Fine-tuning GPT-2

Cell 1: load the dataset

In [6]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
print(dataset)
print(dataset["train"][10])

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})
{'text': ' The game \'s battle system , the BliTZ system , is carried over directly from Valkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters \' turns . Each character has a field and distance of movement limited by their Action Gauge . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting low or being knocked out 

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token   # the article's "padding token" fix

def tokenize_function(examples):
    inputs = tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)
    inputs['labels'] = inputs['input_ids'].copy()
    return inputs

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print(tokenized_datasets)

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3760
    })
})


In [8]:
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)

training_args = TrainingArguments(
    output_dir='/content/results',
    eval_strategy='epoch',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=500,
    save_strategy='no',
    fp16=True,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

Using device: cuda


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [9]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.269377,1.269834


TrainOutput(global_step=9180, training_loss=1.3171136228607112, metrics={'train_runtime': 837.5629, 'train_samples_per_second': 43.839, 'train_steps_per_second': 10.96, 'total_flos': 2398530207744000.0, 'train_loss': 1.3171136228607112, 'epoch': 1.0})

In [10]:
model_output_dir = '/content/results/model'

model.save_pretrained(model_output_dir)
tokenizer.save_pretrained(model_output_dir)
print("Saved to", model_output_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/results/model


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def get_model_parameters(m):
    return sum(p.numel() for p in m.parameters())

def generate(model_path, prompt):
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModelForCausalLM.from_pretrained(model_path).to("cuda")
    print(f"[{model_path}] parameters: {get_model_parameters(mdl)}")
    inputs = tok(prompt, return_tensors='pt').to("cuda")
    outputs = mdl.generate(**inputs, max_new_tokens=50, num_return_sequences=1,
                           pad_token_id=tok.eos_token_id)
    print(tok.decode(outputs[0], skip_special_tokens=True))
    print("-" * 60)

prompt = "The history of the city"

generate('/content/results/model', prompt)   # fine-tuned model
generate('gpt2', prompt)                     # original GPT-2

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[/content/results/model] parameters: 124439808
The history of the city is a complex one . The city 's history is largely a history of the city 's history . The city 's history is largely a history of the city 's history . The city 's history is largely a history of the city '
------------------------------------------------------------


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[gpt2] parameters: 124439808
The history of the city of New York is littered with the stories of the people who lived there. The stories of the people who lived there are often told in the form of stories about the people who lived there.

The story of the city of New York is littered
------------------------------------------------------------


In [12]:
tok = AutoTokenizer.from_pretrained('/content/results/model')
mdl = AutoModelForCausalLM.from_pretrained('/content/results/model').to("cuda")

inputs = tok("The history of the city", return_tensors='pt').to("cuda")
outputs = mdl.generate(**inputs, max_new_tokens=60, do_sample=True,
                       top_k=50, top_p=0.95, temperature=0.8,
                       no_repeat_ngram_size=3, pad_token_id=tok.eos_token_id)
print(tok.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The history of the city has been described as " a city of quiet , peaceful , and noble " , with a " very good " reputation . It has a thriving economy , and the public sector and public health sectors have been described by the mayor as " the most important social initiative in the city " . 



## Kaggle Step 1: Get the dataset into Colab

In [13]:
from google.colab import files
uploaded = files.upload()

Saving Articles.csv to Articles.csv


In [14]:
import pandas as pd

df = pd.read_csv("Articles.csv", encoding="ISO-8859-1")
print(df.shape)
print(df.columns.tolist())
print(df["Article"].iloc[0][:300])

(2692, 4)
['Article', 'Date', 'Heading', 'NewsType']
KARACHI: The Sindh government has decided to bring down public transport fares by 7 per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of tra


## Kaggle Step 2: Clean the text and write Articles.txt

In [15]:
import re

def cleaning(s):
    s = str(s)
    s = re.sub(r'\s\W', ' ', s)
    s = re.sub(r'\W,\s', ' ', s)
    s = re.sub(r"\d+", "", s)
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'[!@#$_]', '', s)
    s = s.replace("co", "")
    s = s.replace("https", "")
    s = s.replace(r"[\w*", " ")
    return s

df = df.dropna()
with open('Articles.txt', 'w') as text_data:
    for idx, item in df.iterrows():
        article = cleaning(item["Article"])
        text_data.write(article)

print("Articles kept after dropna:", df.shape[0])

Articles kept after dropna: 2692


In [16]:
import os

print("File size (MB):", round(os.path.getsize('Articles.txt') / 1e6, 2))
with open('Articles.txt') as f:
    print(f.read(500))

File size (MB): 4.72
KARACHI: The Sindh government has decided to bring down public transport fares by per cent due to massive reduction in petroleum product prices by the federal government, Geo News reported.Sources said reduction in fares will be applicable on public transport, rickshaw, taxi and other means of traveling.Meanwhile, Karachi Transport Ittehad KTI) has refused to abide by the government decision.KTI President Irshad Bukhari said the mmuters are charged the lowest fares in Karachi as mpare to other p


## Kaggle Step 3: Train GPT-2 on the articles

In [17]:
import torch
from torch.utils.data import Dataset
from transformers import DataCollatorForLanguageModeling
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments

# Replacement for the removed TextDataset: reads the file, tokenizes it,
# and cuts it into blocks of block_size tokens
class TextDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size=128):
        with open(file_path, encoding="utf-8") as f:
            text = f.read()
        ids = tokenizer(text)["input_ids"]
        n_blocks = len(ids) // block_size
        self.examples = [ids[i * block_size:(i + 1) * block_size] for i in range(n_blocks)]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return {"input_ids": torch.tensor(self.examples[i], dtype=torch.long)}


def load_text_dataset(file_path, tokenizer, block_size=128):
    return TextDataset(tokenizer=tokenizer, file_path=file_path, block_size=block_size)


def load_data_collator(tokenizer, mlm=False):
    return DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=mlm)


def train(train_file_path, model_name, output_dir,
          per_device_train_batch_size, num_train_epochs, save_steps):
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    train_dataset = load_text_dataset(train_file_path, tokenizer)
    data_collator = load_data_collator(tokenizer)

    tokenizer.save_pretrained(output_dir)

    model = GPT2LMHeadModel.from_pretrained(model_name)
    model.save_pretrained(output_dir)

    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=per_device_train_batch_size,
        num_train_epochs=num_train_epochs,
        save_strategy="no",
        fp16=True,
        logging_steps=500,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
    )

    trainer.train()
    trainer.save_model()

## Cell 2: set the parameters

In [18]:
train_file_path = "Articles.txt"
model_name = 'gpt2'
output_dir = '/content/result'
per_device_train_batch_size = 8
num_train_epochs = 5.0
save_steps = 500

## Cell 3: Train

In [19]:
train(
    train_file_path=train_file_path,
    model_name=model_name,
    output_dir=output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    save_steps=save_steps
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1027107 > 1024). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
500,3.663234
1000,3.389559
1500,3.145347
2000,3.095909
2500,2.956972
3000,2.928126
3500,2.843999
4000,2.816648
4500,2.767427
5000,2.761530


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Kaggle Step 4: Inference

In [20]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

def load_model(model_path):
    return GPT2LMHeadModel.from_pretrained(model_path)

def load_tokenizer(tokenizer_path):
    return GPT2Tokenizer.from_pretrained(tokenizer_path)

def generate_text(sequence, max_length):
    model_path = "/content/result"
    model = load_model(model_path)
    tokenizer = load_tokenizer(model_path)
    ids = tokenizer.encode(f'{sequence}', return_tensors='pt')
    final_outputs = model.generate(
        ids,
        do_sample=True,
        max_length=max_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )
    print(tokenizer.decode(final_outputs[0], skip_special_tokens=True))

In [21]:
sequence = input("Enter a prompt: ")        # try: oil price
max_len = int(input("Max length: "))        # try: 50
generate_text(sequence, max_len)

Enter a prompt: How to become a research engineer?
Max length: 50


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

How to become a research engineer?</strong"The research is already being carried out on more than subjects including the feasibility of autonomous cars, personal care products and the development of biometric data sharing," it added.A team of Pakistani researchers are expected


In [22]:
sequence = "According to the latest report"
max_len = 100

generate_text(sequence, max_len)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

According to the latest report from the Ministry of Finance, the average tax rate on petroleum products is . percent and in other industries it is higher at ..The official said revenue was also going up by Rs billion as mpared to the revenue of . percent for the year.Oil exports to Pakistan are expected to remain at around million barrels of current volume in the year . Acrding to an official, oil exports to Pakistan are expected to be around million barrels in the year .The official said the overall


# Problems

Below are practical problems that can be addressed by fine-tuning GPT-2 on publicly available datasets, along with a brief overview of how to solve each using Google Colab.

---

### 1. Sentiment Analysis on Movie Reviews

**Problem:**  
Automatically classify movie reviews as positive or negative.

**Public Data:**  
IMDb movie reviews dataset (available via Hugging Face Datasets library).

**How to Solve:**
- Load the IMDb dataset using `datasets.load_dataset("imdb")`[3][5].
- Preprocess and tokenize the reviews using GPT-2’s tokenizer.
- Fine-tune GPT-2 (or GPT-2 for sequence classification) for 2–3 epochs on a Colab GPU instance.
- Evaluate the model and use it to predict sentiment for new reviews[3][5].

---


In [23]:
!pip install -q transformers datasets accelerate scikit-learn

In [24]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    GPT2Tokenizer,
    GPT2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [26]:
# dataset = load_dataset("imdb")

# print(dataset)
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [27]:
print(dataset["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [28]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token

In [29]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [30]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

tokenized_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [31]:
model = GPT2ForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=2
)

model.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [32]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [33]:
training_args = TrainingArguments(
    output_dir="./gpt2_imdb",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none"
)

In [34]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.285870,0.290989,0.900160,0.945414,0.849360,0.894817
2,0.168695,0.296132,0.922800,0.923071,0.922480,0.922775


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=0.2697988919067383, metrics={'train_runtime': 3347.4701, 'train_samples_per_second': 14.937, 'train_steps_per_second': 1.867, 'total_flos': 6532418764800000.0, 'train_loss': 0.2697988919067383, 'epoch': 2.0})

In [36]:
results = trainer.evaluate()

print(results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.168695,0.296132,2,0.922800,0.923071,0.922480,0.922775


{'eval_loss': 0.29613175988197327, 'eval_accuracy': 0.9228, 'eval_precision': 0.9230707652897855, 'eval_recall': 0.92248, 'eval_f1': 0.9227752880921894}


In [37]:
trainer.save_model("./gpt2_imdb")
tokenizer.save_pretrained("./gpt2_imdb")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2_imdb/tokenizer_config.json', './gpt2_imdb/tokenizer.json')

In [38]:
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {key: value.to(model.device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=-1).item()

    return "Positive" if prediction == 1 else "Negative"

In [39]:
review = "This movie was amazing. The acting and story were excellent."

print("Review:", review)
print("Sentiment:", predict_sentiment(review))

Review: This movie was amazing. The acting and story were excellent.
Sentiment: Positive


In [40]:
review = "The movie was boring, poorly written, and a complete waste of time."

print("Review:", review)
print("Sentiment:", predict_sentiment(review))

Review: The movie was boring, poorly written, and a complete waste of time.
Sentiment: Negative



### 2. Domain-Specific Text Generation (e.g., Harry Potter Fan Fiction)

**Problem:**  
Generate creative text in the style of a specific domain, such as Harry Potter fan fiction.

**Public Data:**  
Fan fiction or book excerpts from Project Gutenberg or fan sites.

**How to Solve:**
- Gather and clean a collection of relevant text (e.g., Harry Potter books or fan fiction)[4].
- Tokenize the text using GPT-2’s tokenizer.
- Fine-tune GPT-2 for language modeling (text generation) for several epochs.
- Use the model to generate new domain-specific stories or paragraphs[4].

---

In [41]:
# ============================================================
# PROBLEM 2: Domain-specific text generation (fantasy / wizard-style stories)
# Fine-tune GPT-2 on public-domain fantasy books from Project Gutenberg
# ============================================================
import re, math, urllib.request, torch
from torch.utils.data import Dataset, random_split
from transformers import (GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling, set_seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------- SETTINGS ----------------
BOOKS = {                                   # Project Gutenberg ebook IDs
    "Alice's Adventures in Wonderland": 11,
    "Through the Looking-Glass": 12,
    "Peter Pan (Peter and Wendy)": 16,
    "The Wonderful Wizard of Oz": 55,
    "Grimm's Fairy Tales": 2591,
}
CUSTOM_TEXT_FILE = None     # e.g. "/content/my_fanfic.txt" to add your own text
BLOCK_SIZE = 128
EPOCHS = 5
BATCH_SIZE = 8
SAVE_DIR = "/content/fantasy_gpt2"

# ---------------- 1. DOWNLOAD + CLEAN ----------------
def download_book(book_id):
    urls = [f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
            f"https://www.gutenberg.org/ebooks/{book_id}.txt.utf-8"]
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            return urllib.request.urlopen(req, timeout=30).read().decode("utf-8", errors="ignore")
        except Exception:
            pass
    return None

def clean_gutenberg(raw):
    raw = raw.replace("\r\n", "\n")
    m = re.search(r"\*\*\* START OF .*? \*\*\*", raw)      # drop Gutenberg header
    if m: raw = raw[m.end():]
    m = re.search(r"\*\*\* END OF .*? \*\*\*", raw)         # drop Gutenberg footer
    if m: raw = raw[:m.start()]
    raw = re.sub(r"(?<!\n)\n(?!\n)", " ", raw)              # join wrapped lines
    raw = re.sub(r"[ \t]+", " ", raw)
    raw = re.sub(r"\n{3,}", "\n\n", raw)
    return raw.strip()

texts = []
for title, book_id in BOOKS.items():
    raw = download_book(book_id)
    if raw is None:
        print(f"  [skipped] could not download: {title}")
        continue
    t = clean_gutenberg(raw)
    texts.append(t)
    print(f"  [ok] {title}: {len(t):,} characters")

if CUSTOM_TEXT_FILE:
    with open(CUSTOM_TEXT_FILE, encoding="utf-8") as f:
        texts.append(f.read())
    print(f"  [ok] custom file: {CUSTOM_TEXT_FILE}")

if not texts:
    raise RuntimeError("No text loaded. Upload a .txt file to Colab and set CUSTOM_TEXT_FILE.")

# ---------------- 2. TOKENIZE + BUILD DATASET ----------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 10**9          # silences the "sequence too long" warning

corpus = tokenizer.eos_token.join(texts)    # end-of-text token between books
ids = tokenizer(corpus)["input_ids"]
print(f"\nTotal tokens: {len(ids):,}")

class BlockDataset(Dataset):
    def __init__(self, ids, block_size):
        n = len(ids) // block_size
        self.blocks = [ids[i*block_size:(i+1)*block_size] for i in range(n)]
    def __len__(self):
        return len(self.blocks)
    def __getitem__(self, i):
        return {"input_ids": torch.tensor(self.blocks[i], dtype=torch.long)}

full_ds = BlockDataset(ids, BLOCK_SIZE)
n_val = int(0.1 * len(full_ds))             # 90% train / 10% validation
train_ds, val_ds = random_split(full_ds, [len(full_ds) - n_val, n_val],
                                generator=torch.Generator().manual_seed(42))
print(f"Train blocks: {len(train_ds)} | Validation blocks: {len(val_ds)}")

# ---------------- 3. FINE-TUNE GPT-2 ----------------
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="/content/fantasy_results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(model=model, args=args, data_collator=collator,
                  train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

metrics = trainer.evaluate()
print(f"\nValidation loss: {metrics['eval_loss']:.3f} | "
      f"Perplexity: {math.exp(metrics['eval_loss']):.2f}")

# ---------------- 4. SAVE ----------------
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)

# ---------------- 5. GENERATE NEW STORIES ----------------
def generate(m, prompt, n_tokens=100):
    m.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    out = m.generate(**inputs, max_new_tokens=n_tokens, do_sample=True,
                     top_k=50, top_p=0.95, temperature=0.9,
                     no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompts = [
    "The young wizard opened the old door and",
    "Once upon a time, in a castle full of magic,",
    "The witch looked at the little girl and said",
]

print("\n" + "=" * 70)
print("FINE-TUNED MODEL")
print("=" * 70)
for p in prompts:
    print(f"\nPrompt: {p}\n{generate(model, p)}\n" + "-" * 70)

# Comparison with the original, untouched GPT-2
base = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
print("\n" + "=" * 70)
print("BASE GPT-2 (for comparison)")
print("=" * 70)
print(f"\nPrompt: {prompts[0]}\n{generate(base, prompts[0])}")

Using device: cuda
  [ok] Alice's Adventures in Wonderland: 143,949 characters
  [ok] Through the Looking-Glass: 162,795 characters
  [ok] Peter Pan (Peter and Wendy): 255,430 characters
  [ok] The Wonderful Wizard of Oz: 207,668 characters
  [ok] Grimm's Fairy Tales: 519,735 characters

Total tokens: 349,761
Train blocks: 2459 | Validation blocks: 273


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,3.105221,2.848400
2,2.802300,2.792603
3,2.666982,2.773322
4,2.579357,2.768308
5,2.522789,2.770595


Training Loss,Validation Loss,Epoch
2.522789,2.770595,5



Validation loss: 2.771 | Perplexity: 15.97


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/fantasy_gpt2

FINE-TUNED MODEL

Prompt: The young wizard opened the old door and
The young wizard opened the old door and walked in.

“Oh, come in!” he called to the Scarecrow, and when he came in he looked at the old woman, who was sitting on the top of a hill with her eyes shut, and he said, “You haven’t got the right to put your handkerchief over her mouth, have you?”

The woman ran up and said, in a low voice, ‘Dear child, I have the right,
----------------------------------------------------------------------

Prompt: Once upon a time, in a castle full of magic,
Once upon a time, in a castle full of magic, there was an old woman sitting in her garden and singing to herself. “What are you doing here?” she asked.

“Making a dinner for my children,” answered the Queen.
 ““That’s my wish,’ said the Queen; “and in the nursery, we learn to make eggs.”

So the Queen and the other Witches sat down and drank some of the wine. ‘I wish we
-----------------------------

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


BASE GPT-2 (for comparison)

Prompt: The young wizard opened the old door and
The young wizard opened the old door and saw the young wizard standing still. The young wizard, wearing an old cloak, a black cloak and a red hooded cloak, stood in the middle of the room as well. The wizard had a red cloak which he wore. The cloak that had a yellow mark on it had a large yellow ring on it which indicated that the wizard had had some sort of magic spell or event that he could cast. He was also wearing a long-sleeved cloak. The little boy looked into the dark




### 3. FAQ Answer Generation for a Public Dataset

**Problem:**  
Automatically generate answers to frequently asked questions in a specific domain (e.g., COVID-19 FAQs, Stack Overflow programming questions).

**Public Data:**  
- COVID-19 FAQ datasets from official sources.
- Stack Overflow question-answer pairs (public data dumps).

**How to Solve:**
- Collect question-answer pairs and clean the dataset[1].
- Format data as prompt-response pairs for GPT-2.
- Tokenize and fine-tune GPT-2 on these pairs.
- Use the model to generate answers to new, similar questions[1].

---

In [42]:
# ============================================================
# PROBLEM 3: FAQ answer generation (COVID-19 question -> answer)
# Fine-tune GPT-2 on question/answer pairs formatted as prompt + response
# ============================================================
import json, math, random, re, urllib.request, torch
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import (GPT2Tokenizer, GPT2LMHeadModel, Trainer,
                          TrainingArguments, set_seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------- SETTINGS ----------------
MAX_LEN = 192          # max tokens per question+answer example
EPOCHS = 5
BATCH_SIZE = 8
SAVE_DIR = "/content/faq_gpt2"

# ---------------- 1. LOAD QUESTION-ANSWER PAIRS ----------------
def first_answer(ans):
    """Works whether 'answers' is {'text': [...]} or a list of dicts."""
    if isinstance(ans, dict):
        t = ans.get("text", [])
        return t[0] if t else None
    if isinstance(ans, list) and ans:
        a = ans[0]
        return a.get("text") if isinstance(a, dict) else a
    return None

def load_covid_qa():
    # Try Hugging Face first
    for name in ["deepset/covid_qa_deepset", "covid_qa_deepset"]:
        try:
            ds = load_dataset(name, split="train")
            pairs = []
            for r in ds:
                a = first_answer(r["answers"])
                if a:
                    pairs.append((r["question"], a))
            return pairs, f"Hugging Face: {name}"
        except Exception as e:
            print(f"  [note] could not load '{name}' ({type(e).__name__}); trying next source")
    # Fallback: original SQuAD-style JSON on GitHub
    urls = [
        "https://raw.githubusercontent.com/deepset-ai/COVID-QA/master/data/question-answering/COVID-QA.json",
        "https://raw.githubusercontent.com/deepset-ai/COVID-QA/master/data/COVID-QA.json",
    ]
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            data = json.loads(urllib.request.urlopen(req, timeout=60).read().decode("utf-8"))
            pairs = []
            for article in data["data"]:
                for para in article["paragraphs"]:
                    for qa in para["qas"]:
                        if qa.get("answers"):
                            pairs.append((qa["question"], qa["answers"][0]["text"]))
            return pairs, f"GitHub: {url}"
        except Exception as e:
            print(f"  [note] could not load {url} ({type(e).__name__})")
    raise RuntimeError("Could not load COVID-QA from any source.")

pairs, source = load_covid_qa()
clean = lambda s: re.sub(r"\s+", " ", s).strip()
pairs = [(clean(q), clean(a)) for q, a in pairs if q and a]
pairs = list(dict.fromkeys(pairs))          # remove exact duplicates
print(f"Loaded {len(pairs)} question-answer pairs from {source}")
print("Example:", pairs[0])

# ---------------- 2. SPLIT + TOKENIZE ----------------
random.Random(42).shuffle(pairs)
n_val = int(0.1 * len(pairs))               # 90% train / 10% validation
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
print(f"Train: {len(train_pairs)} | Validation: {len(val_pairs)}")

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def make_prompt(q):
    return f"Question: {q}\nAnswer:"

class QADataset(Dataset):
    """Each example = 'Question: ...\\nAnswer: ...<eos>'. Loss is computed on the answer only."""
    def __init__(self, pairs):
        self.items = []
        for q, a in pairs:
            prompt_ids = tokenizer(make_prompt(q))["input_ids"]
            answer_ids = tokenizer(" " + a)["input_ids"] + [tokenizer.eos_token_id]
            if len(prompt_ids) >= MAX_LEN - 5:
                continue
            ids = (prompt_ids + answer_ids)[:MAX_LEN]
            labels = ([-100] * len(prompt_ids) + answer_ids)[:MAX_LEN]
            self.items.append({"input_ids": ids, "labels": labels})
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), maxlen), tokenizer.eos_token_id, dtype=torch.long)
    labels = torch.full((len(batch), maxlen), -100, dtype=torch.long)
    attn = torch.zeros((len(batch), maxlen), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        labels[i, :n] = torch.tensor(b["labels"])
        attn[i, :n] = 1
    return {"input_ids": ids, "attention_mask": attn, "labels": labels}

train_ds, val_ds = QADataset(train_pairs), QADataset(val_pairs)

# ---------------- 3. FINE-TUNE GPT-2 ----------------
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

args = TrainingArguments(
    output_dir="/content/faq_results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=30,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(model=model, args=args, data_collator=collate,
                  train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

metrics = trainer.evaluate()
print(f"\nValidation loss (answers only): {metrics['eval_loss']:.3f} | "
      f"Perplexity: {math.exp(metrics['eval_loss']):.2f}")

# ---------------- 4. SAVE ----------------
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)

# ---------------- 5. GENERATE ANSWERS ----------------
def answer(m, question, max_new=80):
    m.eval()
    inputs = tokenizer(make_prompt(question), return_tensors="pt").to(device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                         no_repeat_ngram_size=3,
                         pad_token_id=tokenizer.eos_token_id,
                         eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()

print("\n" + "=" * 70)
print("HELD-OUT QUESTIONS (not seen during training)")
print("=" * 70)
for q, ref in val_pairs[:3]:
    print(f"\nQ: {q}")
    print(f"Reference answer : {ref}")
    print(f"Model answer     : {answer(model, q)}")
    print("-" * 70)

print("\n" + "=" * 70)
print("YOUR OWN QUESTIONS")
print("=" * 70)
custom_questions = [
    "What are the symptoms of COVID-19?",
    "How does the coronavirus spread between people?",
    "What is the incubation period of the virus?",
]
for q in custom_questions:
    print(f"\nQ: {q}\nA: {answer(model, q)}\n" + "-" * 70)

# Comparison with the original, untouched GPT-2
base = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
print("\n" + "=" * 70)
print("BASE GPT-2 (for comparison)")
print("=" * 70)
print(f"\nQ: {custom_questions[0]}\nA: {answer(base, custom_questions[0])}")

Using device: cuda


README.md:   0%|          | 0.00/5.71k [00:00<?, ?B/s]

covid_qa_deepset/train-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B / 2.27MB            

covid_qa_deepset/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2019 [00:00<?, ? examples/s]

...


Validation loss (answers only): 3.535 | Perplexity: 34.29


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/faq_gpt2

HELD-OUT QUESTIONS (not seen during training)

Q: What is a clinical attack rate?
Reference answer : the proportion of individuals who become ill with or die from a disease in a population initially uninfected
Model answer     : the proportion of patients who are admitted to intensive care or who are symptomatic for at least one of the following reasons:
----------------------------------------------------------------------

Q: Are smokers more likely to contract influenza?
Reference answer : Previous studies have shown that smokers are twice more likely than non-smokers to contract influenza and have more severe symptoms, while smokers were also noted to have higher mortality in the previous MERS-CoV outbreak
Model answer     : smokers are more likely than non-smokers to have had at least one recent influenza infection
----------------------------------------------------------------------

Q: What enzyme is essential for the metabolism of fatty acids?

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


BASE GPT-2 (for comparison)

Q: What are the symptoms of COVID-19?
A: The symptoms of COPD are similar to those of COPDs. COPD is a chronic, chronic, and sometimes fatal disease. COPDs is a disease of the heart, lungs, and liver. COPs is a condition of the brain, which is the part of the body that controls the heart rate. COPS is a disorder of the liver, which controls the blood flow to the brain.




### 4. Headline Generation for News Articles

**Problem:**  
Generate concise headlines for news articles.

**Public Data:**  
CNN/DailyMail news dataset or other open news datasets.

**How to Solve:**
- Download and preprocess article-headline pairs.
- Tokenize the data and fine-tune GPT-2 to generate headlines given article snippets.
- Evaluate model outputs on a validation set.

---


In [43]:
# ============================================================
# PROBLEM 4: Headline generation for news articles
# Fine-tune GPT-2:  "Article: <snippet>\nHeadline:"  ->  headline
# Data: CNN/DailyMail (first highlight line used as the headline)
# ============================================================
import re, math, random, torch
from collections import Counter
from torch.utils.data import Dataset
from datasets import load_dataset
from transformers import (GPT2Tokenizer, GPT2LMHeadModel, Trainer,
                          TrainingArguments, set_seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------- SETTINGS ----------------
N_TRAIN = 4000          # training examples
N_VAL = 300             # validation examples
SNIPPET_WORDS = 100     # use only the first 100 words of each article
MAX_LEN = 256
EPOCHS = 2
BATCH_SIZE = 8
SAVE_DIR = "/content/headline_gpt2"

# ---------------- 1. LOAD + CLEAN ----------------
def clean_article(text):
    text = re.sub(r"^.{0,120}?\(CNN\)\s*(--|-)?\s*", "", text)   # drop "LONDON, England (CNN) -- "
    return re.sub(r"\s+", " ", text).strip()

def make_pair(ex):
    article = ex["article"]
    if "PUBLISHED:" in article[:300] or article.startswith("By ."):   # skip Daily Mail bylines
        return None
    words = clean_article(article).split()
    if len(words) < 60:
        return None
    headline = re.sub(r"\s+", " ", ex["highlights"].split("\n")[0]).strip()
    if not (20 <= len(headline) <= 200):
        return None
    return " ".join(words[:SNIPPET_WORDS]), headline

def load_pairs(split, n):
    for name in ["abisee/cnn_dailymail", "cnn_dailymail"]:
        try:
            stream = load_dataset(name, "3.0.0", split=split, streaming=True)
            pairs = []
            for ex in stream:
                p = make_pair(ex)
                if p:
                    pairs.append(p)
                if len(pairs) >= n:
                    break
            return pairs
        except Exception as e:
            print(f"  [note] streaming '{name}' failed ({type(e).__name__}); trying non-streaming")
            try:
                ds = load_dataset(name, "3.0.0", split=f"{split}[:{n * 4}]")
                pairs = [p for p in (make_pair(ex) for ex in ds) if p]
                return pairs[:n]
            except Exception as e2:
                print(f"  [note] '{name}' failed ({type(e2).__name__})")
    raise RuntimeError("Could not load CNN/DailyMail.")

train_pairs = load_pairs("train", N_TRAIN)
val_pairs = load_pairs("validation", N_VAL)
print(f"Train pairs: {len(train_pairs)} | Validation pairs: {len(val_pairs)}")
print("\nExample snippet :", train_pairs[0][0][:200], "...")
print("Example headline:", train_pairs[0][1])

# ---------------- 2. TOKENIZE ----------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

def make_prompt(snippet):
    return f"Article: {snippet}\nHeadline:"

class HeadlineDataset(Dataset):
    """Loss is computed on the headline tokens only (the prompt is masked)."""
    def __init__(self, pairs):
        self.items = []
        for snippet, headline in pairs:
            prompt_ids = tokenizer(make_prompt(snippet))["input_ids"]
            head_ids = tokenizer(" " + headline)["input_ids"] + [tokenizer.eos_token_id]
            ids = (prompt_ids + head_ids)[:MAX_LEN]
            labels = ([-100] * len(prompt_ids) + head_ids)[:MAX_LEN]
            if all(l == -100 for l in labels):
                continue
            self.items.append({"input_ids": ids, "labels": labels})
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), maxlen), tokenizer.eos_token_id, dtype=torch.long)
    labels = torch.full((len(batch), maxlen), -100, dtype=torch.long)
    attn = torch.zeros((len(batch), maxlen), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        labels[i, :n] = torch.tensor(b["labels"])
        attn[i, :n] = 1
    return {"input_ids": ids, "attention_mask": attn, "labels": labels}

train_ds, val_ds = HeadlineDataset(train_pairs), HeadlineDataset(val_pairs)

# ---------------- 3. FINE-TUNE GPT-2 ----------------
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

args = TrainingArguments(
    output_dir="/content/headline_results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(model=model, args=args, data_collator=collate,
                  train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

metrics = trainer.evaluate()
print(f"\nValidation loss (headline tokens): {metrics['eval_loss']:.3f} | "
      f"Perplexity: {math.exp(metrics['eval_loss']):.2f}")

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)

# ---------------- 4. GENERATE HEADLINES ----------------
def headline(m, snippet, max_new=25):
    m.eval()
    inputs = tokenizer(make_prompt(snippet), return_tensors="pt").to(device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                         no_repeat_ngram_size=3,
                         pad_token_id=tokenizer.eos_token_id,
                         eos_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text.strip().split("\n")[0].strip()

# ---------------- 5. EVALUATE ON VALIDATION SET (ROUGE-1 and ROUGE-L) ----------------
def toks(s):
    return re.findall(r"\w+", s.lower())

def rouge1(pred, ref):
    p, r = toks(pred), toks(ref)
    overlap = sum((Counter(p) & Counter(r)).values())
    if not p or not r or overlap == 0:
        return 0.0
    pr, rc = overlap / len(p), overlap / len(r)
    return 2 * pr * rc / (pr + rc)

def lcs_len(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a)):
        for j in range(len(b)):
            dp[i + 1][j + 1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j + 1], dp[i + 1][j])
    return dp[-1][-1]

def rougeL(pred, ref):
    p, r = toks(pred), toks(ref)
    l = lcs_len(p, r) if p and r else 0
    if l == 0:
        return 0.0
    pr, rc = l / len(p), l / len(r)
    return 2 * pr * rc / (pr + rc)

def evaluate_rouge(m, pairs, n=100):
    r1, rl = [], []
    for snippet, ref in pairs[:n]:
        pred = headline(m, snippet)
        r1.append(rouge1(pred, ref))
        rl.append(rougeL(pred, ref))
    return sum(r1) / len(r1), sum(rl) / len(rl)

base = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
ft_r1, ft_rl = evaluate_rouge(model, val_pairs)
base_r1, base_rl = evaluate_rouge(base, val_pairs)
print("\n" + "=" * 70)
print("ROUGE F1 on 100 validation articles")
print("=" * 70)
print(f"Fine-tuned GPT-2 : ROUGE-1 = {ft_r1:.3f} | ROUGE-L = {ft_rl:.3f}")
print(f"Base GPT-2       : ROUGE-1 = {base_r1:.3f} | ROUGE-L = {base_rl:.3f}")

# ---------------- 6. EXAMPLES ----------------
print("\n" + "=" * 70)
print("VALIDATION EXAMPLES")
print("=" * 70)
for snippet, ref in val_pairs[:5]:
    print(f"\nArticle start : {snippet[:220]}...")
    print(f"Reference     : {ref}")
    print(f"Model headline: {headline(model, snippet)}")
    print("-" * 70)

print("\n" + "=" * 70)
print("YOUR OWN ARTICLE")
print("=" * 70)
custom_article = (
    "The city council voted on Tuesday to approve a new plan to expand the metro rail "
    "network, adding three lines and twelve stations over the next eight years. Officials "
    "said the project will cost about 50 million dollars and is expected to reduce traffic "
    "congestion and cut commuting times for thousands of residents. Construction on the "
    "first line is scheduled to begin early next year."
)
print("Headline:", headline(model, custom_article))

Using device: cuda


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

Train pairs: 4000 | Validation pairs: 300

Example snippet : LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on  ...
Example headline: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,2.357635,2.349185
2,1.806720,2.404541


...


Validation loss (headline tokens): 2.405 | Perplexity: 11.07


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/headline_gpt2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


ROUGE F1 on 100 validation articles
Fine-tuned GPT-2 : ROUGE-1 = 0.169 | ROUGE-L = 0.153
Base GPT-2       : ROUGE-1 = 0.141 | ROUGE-L = 0.124

VALIDATION EXAMPLES

Article start : Share, and your gift will be multiplied. That may sound like an esoteric adage, but when Zully Broussard selflessly decided to give one of her kidneys to a stranger, her generosity paired up with big data. It resulted in...
Reference     : Zully Broussard decided to give a kidney to a stranger .
Model headline: Zully donated kidney to a woman who had kidney cancer .
----------------------------------------------------------------------

Article start : On the 6th of April 1996, San Jose Clash and DC United strode out in front of 31,683 expectant fans at the Spartan Stadium in San Jose, California. The historic occasion was the first ever Major League Soccer match -- a ...
Reference     : The 20th MLS season begins this weekend .
Model headline: NEW: "It's a dream come true"
----------------------------------



### 5. Poetry or Song Lyric Generation

**Problem:**  
Generate poetry or song lyrics in a particular style.

**Public Data:**  
Poetry Foundation corpus, Project Gutenberg poetry, or song lyrics datasets.

**How to Solve:**
- Gather and clean a corpus of poems or lyrics.
- Tokenize and fine-tune GPT-2 for text generation.
- Prompt the model with a first line or theme to generate new poetic content.

---

In [44]:
# ============================================================
# PROBLEM 5: Poetry generation
# Fine-tune GPT-2 on public-domain poetry from Project Gutenberg
# ============================================================
import re, math, urllib.request, torch
from torch.utils.data import Dataset, random_split
from transformers import (GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling, set_seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------- SETTINGS ----------------
# (Gutenberg ebook ID, keyword that MUST appear in the book's title - a safety check)
BOOKS = [
    (1322, "Leaves of Grass"),     # Walt Whitman
    (2678, "Dickinson"),           # Emily Dickinson, Series One
    (2679, "Dickinson"),           # Emily Dickinson, Series Two (skipped if the check fails)
    (1041, "Sonnets"),             # Shakespeare's Sonnets (skipped if the check fails)
    (1934, "Innocence"),           # Blake, Songs of Innocence and of Experience
    (1065, "Raven"),               # Poe, The Raven
]
MAX_CHARS_PER_BOOK = 250_000       # stops one long book (Whitman) from dominating
CUSTOM_TEXT_FILE = None            # e.g. "/content/my_poems.txt" to add your own text
BLOCK_SIZE = 128
EPOCHS = 4
BATCH_SIZE = 8
SAVE_DIR = "/content/poetry_gpt2"

# ---------------- 1. DOWNLOAD + CLEAN ----------------
def download_book(book_id):
    urls = [f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
            f"https://www.gutenberg.org/ebooks/{book_id}.txt.utf-8"]
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            return urllib.request.urlopen(req, timeout=30).read().decode("utf-8", errors="ignore")
        except Exception:
            pass
    return None

def get_title(raw):
    m = re.search(r"^Title:\s*(.+)$", raw, flags=re.M)
    return m.group(1).strip() if m else ""

def clean_poetry(raw):
    raw = raw.replace("\r\n", "\n")
    m = re.search(r"\*\*\* ?START OF .*? ?\*\*\*", raw)     # drop Gutenberg header
    if m: raw = raw[m.end():]
    m = re.search(r"\*\*\* ?END OF .*? ?\*\*\*", raw)        # drop Gutenberg footer
    if m: raw = raw[:m.start()]
    raw = re.sub(r"\[Illustration[^\]]*\]", "", raw)
    raw = "\n".join(line.rstrip() for line in raw.split("\n"))
    raw = re.sub(r"\n{3,}", "\n\n", raw)                     # keep line breaks (they matter in poetry)
    return raw.strip()

texts = []
for book_id, keyword in BOOKS:
    raw = download_book(book_id)
    if raw is None:
        print(f"  [skipped] #{book_id}: could not download")
        continue
    title = get_title(raw)
    if keyword.lower() not in title.lower():
        print(f"  [skipped] #{book_id}: title '{title}' does not match '{keyword}'")
        continue
    t = clean_poetry(raw)[:MAX_CHARS_PER_BOOK]
    texts.append(t)
    print(f"  [ok] #{book_id}: {title} ({len(t):,} characters used)")

if CUSTOM_TEXT_FILE:
    with open(CUSTOM_TEXT_FILE, encoding="utf-8") as f:
        texts.append(f.read())
    print(f"  [ok] custom file: {CUSTOM_TEXT_FILE}")

if not texts:
    raise RuntimeError("No poetry loaded. Upload a .txt file and set CUSTOM_TEXT_FILE.")

# ---------------- 2. TOKENIZE + BUILD DATASET ----------------
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 10**9

corpus = tokenizer.eos_token.join(texts)
ids = tokenizer(corpus)["input_ids"]
print(f"\nTotal tokens: {len(ids):,}")

class BlockDataset(Dataset):
    def __init__(self, ids, block_size):
        n = len(ids) // block_size
        self.blocks = [ids[i * block_size:(i + 1) * block_size] for i in range(n)]
    def __len__(self):
        return len(self.blocks)
    def __getitem__(self, i):
        return {"input_ids": torch.tensor(self.blocks[i], dtype=torch.long)}

full_ds = BlockDataset(ids, BLOCK_SIZE)
n_val = int(0.1 * len(full_ds))
train_ds, val_ds = random_split(full_ds, [len(full_ds) - n_val, n_val],
                                generator=torch.Generator().manual_seed(42))
print(f"Train blocks: {len(train_ds)} | Validation blocks: {len(val_ds)}")

# ---------------- 3. FINE-TUNE GPT-2 ----------------
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="/content/poetry_results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(model=model, args=args, data_collator=collator,
                  train_dataset=train_ds, eval_dataset=val_ds)
trainer.train()

metrics = trainer.evaluate()
print(f"\nValidation loss: {metrics['eval_loss']:.3f} | "
      f"Perplexity: {math.exp(metrics['eval_loss']):.2f}")

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Model saved to", SAVE_DIR)

# ---------------- 4. GENERATE POETRY ----------------
def write_poem(m, prompt, n_tokens=80):
    m.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=n_tokens, do_sample=True,
                         top_k=50, top_p=0.95, temperature=0.9,
                         no_repeat_ngram_size=3,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

prompts = [
    "The moon rose slowly over the sea,\n",
    "Love is a\n",
    "Winter came, and\n",
    "Upon the hill\n",
]

print("\n" + "=" * 70)
print("FINE-TUNED MODEL")
print("=" * 70)
for p in prompts:
    print(f"\n--- Prompt: {p.strip()!r} ---\n{write_poem(model, p)}\n")

# Comparison with the original GPT-2
base = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
print("=" * 70)
print("BASE GPT-2 (for comparison)")
print("=" * 70)
print(f"\n--- Prompt: {prompts[0].strip()!r} ---\n{write_poem(base, prompts[0])}")

Using device: cuda
  [ok] #1322: Leaves of Grass (250,000 characters used)
  [ok] #2678: Poems by Emily Dickinson, Series One (49,587 characters used)
  [ok] #2679: Poems by Emily Dickinson, Series Two (74,941 characters used)
  [ok] #1041: Shakespeare's Sonnets (96,208 characters used)
  [ok] #1934: Songs of Innocence and of Experience (30,906 characters used)
  [ok] #1065: The Raven (7,116 characters used)

Total tokens: 151,052
Train blocks: 1062 | Validation blocks: 118


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,4.011169,3.607629
2,3.557426,3.558982
3,3.405121,3.556051
4,3.325700,3.554425


Training Loss,Validation Loss,Epoch
3.325700,3.554425,4



Validation loss: 3.554 | Perplexity: 34.97


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/poetry_gpt2

FINE-TUNED MODEL

--- Prompt: 'The moon rose slowly over the sea,' ---
The moon rose slowly over the sea,
And made haste to leave the sick
That were dead.

    The morning brought the worst,
The night was calm and calm.


LITTLE HUMPER

We must not let go,
Our lips should not feel the cold,
We shall not let ourselves be seen,
Or heard or heard any thing.

That is to be, --



--- Prompt: 'Love is a' ---
Love is a
Unseen love,
And the joy to have it, --
    But I see it not.

   My heart is my soul,

But what I do is that of another.

For what you see is what you hear,
  Not my voice, not the thing
Which I might hear!

  I am not the body,
But the


--- Prompt: 'Winter came, and' ---
Winter came, and
    But I had a dream.

I was at my bedside,
  And the birds, with their sharp wings,

  Were playing,
I wondered if there was anything,
The birds,
Which I never thought
  To play with.
But all the birds had their
  Eyes open,
And I did not see
  They we

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

BASE GPT-2 (for comparison)

--- Prompt: 'The moon rose slowly over the sea,' ---
The moon rose slowly over the sea,

Than the stars of the moon had no stars,
, because the stars were not in the sea.

The waters of the earth rose up out of the ocean,
. . . ,

and the stars rose out of it.
. And they rose out from the sea;

for the stars had no part in the world;
 (for the world was
